# Save Qwen3-8B to Kaggle Dataset — Run Once

Downloads Qwen3-8B weights from HuggingFace and pushes them as a Kaggle dataset.
After this runs once, all 3 agent kernels load from `/kaggle/input/qwen3-8b-cache/`
instead of re-downloading every run.

**Time:** ~15 min (download) + ~5 min (upload to Kaggle)
**Run once per account — never again.**

**After running:**
1. Go to kaggle.com/datasets/builder117/qwen3-8b-cache
2. Add it as a Data Source to each agent kernel
3. Future agent runs load in ~2 min instead of ~15 min

In [ ]:
import subprocess, os
# Ensure HF token available
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN", "")

subprocess.run(["pip", "install", "-q", "huggingface_hub>=0.23.0"], check=True)
print("Install complete. HF_TOKEN set:", bool(HF_TOKEN))


In [ ]:
import os
from pathlib import Path
from huggingface_hub import snapshot_download

HF_TOKEN = os.environ.get("HF_TOKEN", "")
KAGGLE_USERNAME = "amatullahvakhariya"
MODEL_ID = "Qwen/Qwen3-8B"
SAVE_DIR = Path("/kaggle/working/qwen3-8b-cache")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Downloading {MODEL_ID} to {SAVE_DIR} ...")
print("Takes ~15 min on Kaggle. Watch disk usage in right panel.")

snapshot_download(
    repo_id=MODEL_ID,
    local_dir=str(SAVE_DIR),
    token=HF_TOKEN if HF_TOKEN else None,
    ignore_patterns=["*.msgpack", "*.h5", "flax_*", "tf_*", "*.ot", "rust_model.ot"],
)

files = list(SAVE_DIR.rglob("*"))
total_gb = sum(f.stat().st_size for f in files if f.is_file()) / 1e9
print(f"Downloaded {len(files)} files, {total_gb:.1f} GB")


In [ ]:
import json
from pathlib import Path

KAGGLE_USERNAME = "amatullahvakhariya"
SAVE_DIR = Path("/kaggle/working/qwen3-8b-cache")

meta = {
    "title": "Qwen3-8B Model Cache",
    "id": f"{KAGGLE_USERNAME}/qwen3-8b-cache",
    "licenses": [{"name": "apache-2.0"}],
    "description": "Qwen3-8B weights for fast agent kernel loads. INT4 quantization via BitsAndBytesConfig nf4."
}
(SAVE_DIR / "dataset-metadata.json").write_text(json.dumps(meta, indent=2))
print(f"Metadata written: {KAGGLE_USERNAME}/qwen3-8b-cache")


In [ ]:
import subprocess
from pathlib import Path

SAVE_DIR = Path("/kaggle/working/qwen3-8b-cache")

print("Pushing dataset to Kaggle (may take ~5 min for 16GB)...")
result = subprocess.run(
    ["kaggle", "datasets", "create", "-p", str(SAVE_DIR)],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)
    # Already exists — push a new version instead
    result2 = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(SAVE_DIR), "-m", "initial weights"],
        capture_output=True, text=True
    )
    print(result2.stdout)
    if result2.stderr:
        print("STDERR:", result2.stderr)
else:
    print("Done!")
    print("Dataset: kaggle.com/datasets/amatullahvakhariya/qwen3-8b-cache")
    print()
    print("trigger_agents.py will now attach it automatically via dataset_data_sources.")
    print("Agent kernels load from /kaggle/input/qwen3-8b-cache/ in ~2 min.")
